<a href="https://colab.research.google.com/github/handdung/bigdata/blob/main/week05_5756498_%EC%9D%B4%ED%99%98%EC%A4%80_ipynb%EC%9D%98_%EC%82%AC%EB%B3%B8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 5주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week05_학번_이름.ipynb` 로 제출합니다.


In [1]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

torch.manual_seed(0)


## 문제 1. 사용자 정의 Dataset 작성 (40점)
아래 조건으로 Dataset 클래스를 작성하세요.

- 데이터 개수는 30개, 입력 특성은 4개
- `__init__` `__len__` `__getitem__` 세 메서드를 모두 구현
- 각 메서드가 하는 일을 주석으로 한 줄씩 설명


In [2]:
# 답안
class MyDataset(Dataset):
    def __init__(self, n_samples=30, n_features=4):
        # 데이터 30개 × 특성 4개를 만들고, 정답 y를 선형식 + 잡음으로 생성해 저장한다
        self.X = torch.randn(n_samples, n_features)
        true_w = torch.tensor([[2.0], [-1.0], [0.5], [3.0]])
        self.y = self.X @ true_w + 1.0 + 0.1 * torch.randn(n_samples, 1)

    def __len__(self):
        # 데이터셋에 들어 있는 샘플의 전체 개수(30)를 돌려준다
        return len(self.X)

    def __getitem__(self, idx):
        # idx번째 샘플의 (입력, 정답) 한 쌍을 돌려준다
        return self.X[idx], self.y[idx]

dataset = MyDataset()
print("데이터 개수:", len(dataset))
x0, y0 = dataset[0]
print("첫 샘플 입력 shape:", x0.shape, "/ 정답 shape:", y0.shape)

데이터 개수: 30
첫 샘플 입력 shape: torch.Size([4]) / 정답 shape: torch.Size([1])


## 문제 2. DataLoader로 batch 출력 (30점)
문제 1의 Dataset을 DataLoader에 넣고 아래를 출력하세요.

- batch 개수
- 각 batch의 입력 shape과 정답 shape
- 마지막 batch의 크기가 다른 이유 (batch size를 7로 두고 확인)


In [3]:
# 답안
loader = DataLoader(dataset, batch_size=7, shuffle=True)

print("batch 개수:", len(loader))
for i, (xb, yb) in enumerate(loader):
    print(f"batch {i+1}: 입력 shape {tuple(xb.shape)}, 정답 shape {tuple(yb.shape)}")

print(f"\n마지막 batch 크기가 다른 이유: 30 = 7×4 + 2 이므로 "
      f"7개짜리 batch 4개를 만들고 남은 {len(dataset) % 7}개가 마지막 batch가 된다.")

batch 개수: 5
batch 1: 입력 shape (7, 4), 정답 shape (7, 1)
batch 2: 입력 shape (7, 4), 정답 shape (7, 1)
batch 3: 입력 shape (7, 4), 정답 shape (7, 1)
batch 4: 입력 shape (7, 4), 정답 shape (7, 1)
batch 5: 입력 shape (2, 4), 정답 shape (2, 1)

마지막 batch 크기가 다른 이유: 30 = 7×4 + 2 이므로 7개짜리 batch 4개를 만들고 남은 2개가 마지막 batch가 된다.


## 문제 3. batch shape 설명 (10점)
문제 2의 출력에서 batch shape이 왜 그렇게 나오는지 설명하세요.
`(batch_size, 특성 수)` 의 각 숫자가 무엇을 뜻하는지 적으면 됩니다.


**답:**


## 문제 4. 학습 루프 작성 (20점)
문제 1의 Dataset으로 아래를 수행하세요.

1. `nn.Linear` 모델 하나를 만든다
2. 다섯 단계(예측 → 손실 → 초기화 → 역전파 → 수정)로 학습 루프를 작성한다
3. epoch마다 평균 손실을 출력해 손실이 내려가는지 확인한다


In [4]:
# 답안
model = nn.Linear(4, 1)                       # 입력 특성 4개 → 출력 1개
loss_fn = nn.MSELoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)

for epoch in range(1, 21):
    total_loss = 0.0
    for xb, yb in loader:
        pred = model(xb)                      # 1. 예측
        loss = loss_fn(pred, yb)              # 2. 손실 계산
        optimizer.zero_grad()                 # 3. 기울기 초기화
        loss.backward()                       # 4. 역전파
        optimizer.step()                      # 5. 가중치 수정
        total_loss += loss.item() * len(xb)   # batch 크기를 반영해 누적

    avg_loss = total_loss / len(dataset)
    print(f"epoch {epoch:2d} | 평균 손실 {avg_loss:.4f}")

epoch  1 | 평균 손실 10.7848
epoch  2 | 평균 손실 2.6690
epoch  3 | 평균 손실 1.1962
epoch  4 | 평균 손실 0.4252
epoch  5 | 평균 손실 0.1486
epoch  6 | 평균 손실 0.0709
epoch  7 | 평균 손실 0.0358
epoch  8 | 평균 손실 0.0236
epoch  9 | 평균 손실 0.0198
epoch 10 | 평균 손실 0.0172
epoch 11 | 평균 손실 0.0158
epoch 12 | 평균 손실 0.0157
epoch 13 | 평균 손실 0.0147
epoch 14 | 평균 손실 0.0147
epoch 15 | 평균 손실 0.0147
epoch 16 | 평균 손실 0.0145
epoch 17 | 평균 손실 0.0140
epoch 18 | 평균 손실 0.0140
epoch 19 | 평균 손실 0.0135
epoch 20 | 평균 손실 0.0139
